In [ ]:
# ============================================================================
#  Colab LLM Server  ·  Ollama -> LiteLLM Proxy -> Cloudflare Tunnel
#  Runtime -> Ubah tipe runtime -> GPU T4, lalu jalankan sel ini.
#  Mencatat setiap langkah; jika gagal, akan mencetak log yang relevan sehingga dapat melihat kesalahannya.
# ============================================================================
import subprocess, time, os, re, shutil, requests, datetime

# ── CONFIG ──────────────────────────────────────────────────────────────────
# Text model (default). Hapus tanda komentar pada baris lain untuk beralih:
# MODEL       = "qwen2.5:7b-instruct"                 # Cepat, serba bisa
# MODEL     = "qwen2.5:14b-instruct"                # Lebih cerdas, download yang lebih besar
# MODEL     = "qwen2.5vl:7b"                         # VISION: bisa melihat gambar (7B, cepat)
# MODEL     = "hf.co/mradermacher/EVA-abliterated-TIES-Qwen2.5-14B-i1-GGUF:Q6_K"  # VISION: bisa melihat gambar (14B, lebih lambat)
MODEL = "hf.co/yuxinlu1/gemma-4-12B-agentic-fable5-composer2.5-v2-3.5x-tau2-GGUF:Q4_K_M" 

PUBLIC_NAME = "character1"      # name your app requests (MODEL_NAME in .env)
API_KEY     = "sk-colab-local"  # harus sesuai dengan LLM_API_KEY di .env
OLLAMA_PORT = 11434
PROXY_PORT  = 4000
NUM_CTX     = 8192              # context window (tokens)
# ────────────────────────────────────────────────────────────────────────────

def log(msg, symbol="•"):
    ts = datetime.datetime.now().strftime("%H:%M:%S")
    print(f"  [{ts}] {symbol} {msg}", flush=True)

def run(cmd, live=False):
    """Jalankan perintah shell. Mencetak output jika gagal (atau selalu, jika berhasil dijalankan)."""
    if live:
        return subprocess.run(cmd, shell=True)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        log(f"perintah gagal: {cmd}", "✗")
        if r.stdout.strip(): print("     stdout:", r.stdout.strip()[-600:])
        if r.stderr.strip(): print("     stderr:", r.stderr.strip()[-600:])
    return r

def bg(cmd, logfile):
    return subprocess.Popen(cmd, shell=True, stdout=open(logfile, "w"), stderr=subprocess.STDOUT)

def tail(path, n=25):
    try:
        lines = open(path).read().splitlines()
        return "\n".join(lines[-n:])
    except Exception as e:
        return f"(tidak bisa membaca {path}: {e})"

def wait_http(url, timeout=180, name=""):
    for i in range(timeout):
        try:
            if requests.get(url, timeout=2).status_code < 500:
                log(f"{name} sudah siap", "✅"); return True
        except Exception:
            pass
        if i and i % 20 == 0:
            log(f"masih menunggu {name}… ({i}s)", "⏳")
        time.sleep(1)
    log(f"{name} TIDAK siap setelah {timeout}s", "✗")
    return False

def die(msg, logpath=None):
    log(msg, "🛑")
    if logpath:
        print("\n  ── baris log terakhir ──")
        print(tail(logpath))
    raise SystemExit(f"Berhenti: {msg}")

def section(title):
    print("\n" + "─" * 60)
    print(f"  {title}")
    print("─" * 60)

# ── Banner + GPU check ───────────────────────────────────────────────────────
print("=" * 60)
print("  Colab LLM Server  ·  Ollama -> LiteLLM -> Cloudflare")
print("=" * 60)
gpu = run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
if gpu.returncode == 0 and gpu.stdout.strip():
    log(f"GPU: {gpu.stdout.strip()}", "🖥")
else:
    log("GPU tidak terdeteksi — aktifkan: Runtime -> Ubah tipe runtime -> GPU T4", "⚠️")
log(f"Model: {MODEL}", "🤖")

# Bersihkan sisa-sisa jika sel dijalankan ulang.
run("pkill -f 'ollama serve'; pkill -f litellm; pkill -f cloudflared"); time.sleep(2)

# ── 1. Instal Ollama (GitHub .tar.zst — andal di Colab)──────────────────
section("[1/6] Install Ollama")
run("apt-get install -y -qq zstd")
run("rm -f /content/ollama.tar.zst")
log("mengunduh Ollama dari GitHub…")
run("curl -fL -o /content/ollama.tar.zst "
    "https://github.com/ollama/ollama/releases/latest/download/ollama-linux-amd64.tar.zst")
size = run("stat -c%s /content/ollama.tar.zst")
log(f"ukuran unduhan: {size.stdout.strip()} bytes")
run("tar --zstd -xf /content/ollama.tar.zst -C /usr")
OLLAMA = shutil.which("ollama") or ("/usr/bin/ollama" if os.path.exists("/usr/bin/ollama") else None)
if not OLLAMA:
    die("Instalasi Ollama gagal (biner tidak ditemukan). Periksa ukuran unduhan di atas — 0 byte = masalah jaringan.")
log(f"ollama binary: {OLLAMA}", "✅")

# ── 2. Install LiteLLM + cloudflared ─────────────────────────────────────────
section("[2/6] Install LiteLLM + cloudflared")
log("pip installing litellm (quiet)…")
run("pip -q install 'litellm[proxy]'")
run("curl -fL -o /usr/bin/cloudflared "
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
    "&& chmod +x /usr/bin/cloudflared")
log("dependensi terinstal", "✅")

# ── 3. Start Ollama server ───────────────────────────────────────────────────
section("[3/6] Start Ollama server")
os.environ["OLLAMA_HOST"] = f"0.0.0.0:{OLLAMA_PORT}"
bg(f"{OLLAMA} serve", "/content/ollama.log")
if not wait_http(f"http://localhost:{OLLAMA_PORT}", 60, "Ollama"):
    die("Ollama server did not start.", "/content/ollama.log")

# ── 4. Pull the model ────────────────────────────────────────────────────────
section("[4/6] Pull model (kemajuan ditampilkan secara langsung)")
log(f"pulling {MODEL} — Unduhan pertama berukuran beberapa GB…")
pull = run(f"{OLLAMA} pull {MODEL}", live=True)
if pull.returncode != 0:
    die(f"Gagal pull '{MODEL}'. Periksa apakah nama/label model sudah benar.")
# Alias with a fixed context window so long chats aren't silently truncated.
open("/content/Modelfile", "w").write(f"FROM {MODEL}\nPARAMETER num_ctx {NUM_CTX}\n")
alias = run("{o} create character -f /content/Modelfile".format(o=OLLAMA))
OLLAMA_MODEL = "character" if alias.returncode == 0 else MODEL
log(f"menggunakan model ollama: {OLLAMA_MODEL}", "✅")

# ── 5. Start LiteLLM proxy ───────────────────────────────────────────────────
section("[5/6] Start LiteLLM proxy")
open("/content/litellm.yaml", "w").write(f"""
model_list:
  - model_name: {PUBLIC_NAME}
    litellm_params:
      model: ollama/{OLLAMA_MODEL}
      api_base: http://localhost:{OLLAMA_PORT}
general_settings:
  master_key: {API_KEY}
litellm_settings:
  drop_params: true
""")
bg(f"litellm --config /content/litellm.yaml --port {PROXY_PORT} --host 0.0.0.0", "/content/litellm.log")
if not wait_http(f"http://localhost:{PROXY_PORT}/health/liveliness", 90, "LiteLLM proxy"):
    die("LiteLLM proxy did not start.", "/content/litellm.log")

# ── 6. Cloudflare tunnel ─────────────────────────────────────────────────────
section("[6/6] Open Cloudflare tunnel")
bg(f"cloudflared tunnel --url http://localhost:{PROXY_PORT} --no-autoupdate", "/content/cloudflared.log")
url = None
for _ in range(40):
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", tail("/content/cloudflared.log", 100))
    if m: url = m.group(0); break
    time.sleep(1)
if not url:
    die("Could not get a tunnel URL.", "/content/cloudflared.log")
log("tunnel is up", "✅")

# ── Done ─────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  ✅  SERVER SIAP — tempelkan ini ke dalam file .env lokal Anda:")
print("=" * 60)
print(f"  BASE_URL={url}")
print(f"  LLM_API_KEY={API_KEY}")
print(f"  MODEL_NAME={PUBLIC_NAME}")
print("=" * 60)
print("  Pertahankan sel ini tetap BERJALAN. Untuk menghentikan: Runtime -> Putuskan sambungan dan hapus runtime.")
print("  URL berubah setiap kali restart — perbarui file .env setiap kali.\n")